## Insert Overwrite
1. Replace all the data in a table
1. Replace all the data from a specific partition
1. How to handle schema changes

INSERT OVERWITE - Overwrites the existing data in a table or a specific partition with the new data. 

INSERT INTO - Appends new data


### 1. Replace all the data in a table

In [0]:
DROP TABLE IF EXISTS demo.delta_lake.gold_companies;

CREATE TABLE demo.delta_lake.gold_companies
  (company_name STRING,
   founded_date DATE,
   country      STRING);

INSERT INTO demo.delta_lake.gold_companies 
(company_name, founded_date, country)
VALUES ("Apple", "1976-04-01", "USA"),  
       ("Tencent", "1998-11-11", "China"); 

SELECT * FROM demo.delta_lake.gold_companies;        

company_name,founded_date,country
Apple,1976-04-01,USA
Tencent,1998-11-11,China


In [0]:
DROP TABLE IF EXISTS demo.delta_lake.bronze_companies;

CREATE TABLE demo.delta_lake.bronze_companies
  (company_name STRING,
   founded_date DATE,
   country      STRING);

INSERT INTO demo.delta_lake.bronze_companies 
(company_name, founded_date, country)
VALUES ("Apple", "1976-04-01", "USA"),
       ("Microsoft", "1975-04-04", "USA"),
       ("Google", "1998-09-04", "USA"),
       ("Amazon", "1994-07-05", "USA"),
       ("Tencent", "1998-11-11", "China");   

SELECT * FROM demo.delta_lake.bronze_companies;       

company_name,founded_date,country
Apple,1976-04-01,USA
Microsoft,1975-04-04,USA
Google,1998-09-04,USA
Amazon,1994-07-05,USA
Tencent,1998-11-11,China


In [0]:
INSERT OVERWRITE TABLE demo.delta_lake.gold_companies
SELECT *
  FROM demo.delta_lake.bronze_companies;

num_affected_rows,num_inserted_rows
5,5


In [0]:
SELECT * FROM demo.delta_lake.gold_companies;

company_name,founded_date,country
Apple,1976-04-01,USA
Microsoft,1975-04-04,USA
Google,1998-09-04,USA
Amazon,1994-07-05,USA
Tencent,1998-11-11,China


In [0]:
DESC HISTORY demo.delta_lake.gold_companies;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-01-08T17:31:43Z,3341927249304550,cloudboxacademy@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(4070074947181975),1108-132705-qy6b0rb7,1,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1098)",null,Databricks-Runtime/15.4.x-scala2.12
1,2025-01-08T17:29:46Z,3341927249304550,cloudboxacademy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(4070074947181975),1108-132705-qy6b0rb7,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1009)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-01-08T17:29:25Z,3341927249304550,cloudboxacademy@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(4070074947181975),1108-132705-qy6b0rb7,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-scala2.12


### 2. Replace all the data from a specific partition

In [0]:
DROP TABLE IF EXISTS demo.delta_lake.gold_companies_partitioned;

CREATE TABLE demo.delta_lake.gold_companies_partitioned
  (company_name STRING,
   founded_date DATE,
   country      STRING)
PARTITIONED BY (country);

INSERT INTO demo.delta_lake.gold_companies_partitioned 
(company_name, founded_date, country)
VALUES ("Apple", "1976-04-01", "USA"),  
       ("Tencent", "1998-11-11", "China"); 

SELECT * FROM demo.delta_lake.gold_companies_partitioned;        

company_name,founded_date,country
Tencent,1998-11-11,China
Apple,1976-04-01,USA


In [0]:
DESC EXTENDED demo.delta_lake.gold_companies_partitioned

col_name,data_type,comment
company_name,string,null
founded_date,date,null
country,string,null
# Partition Information,,
# col_name,data_type,comment
country,string,null
,,
# Delta Statistics Columns,,
Column Names,"company_name, founded_date",
Column Selection Method,first-32,


In [0]:
DROP TABLE IF EXISTS demo.delta_lake.bronze_companies_usa;

CREATE TABLE demo.delta_lake.bronze_companies_usa
  (company_name STRING,
   founded_date DATE,
   country      STRING);

INSERT INTO demo.delta_lake.bronze_companies_usa 
(company_name, founded_date, country)
VALUES ("Apple", "1976-04-01", "USA"),
       ("Microsoft", "1975-04-04", "USA"),
       ("Google", "1998-09-04", "USA"),
       ("Amazon", "1994-07-05", "USA");   

SELECT * FROM demo.delta_lake.bronze_companies_usa;       

company_name,founded_date,country
Apple,1976-04-01,USA
Microsoft,1975-04-04,USA
Google,1998-09-04,USA
Amazon,1994-07-05,USA


In [0]:
INSERT OVERWRITE TABLE demo.delta_lake.gold_companies_partitioned
PARTITION (country = "USA")
SELECT company_name,
       founded_date
  FROM demo.delta_lake.bronze_companies_usa;     

num_affected_rows,num_inserted_rows
4,4


In [0]:
SELECT * FROM demo.delta_lake.gold_companies_partitioned;

company_name,founded_date,country
Apple,1976-04-01,USA
Microsoft,1975-04-04,USA
Google,1998-09-04,USA
Amazon,1994-07-05,USA
Tencent,1998-11-11,China


### 3. How to handle schema changes

Insert Overwrite -> Use to overwrite the data in a table or a partition when there are no schema changes.  
Create or replace table -> Use when there are schema changes. 
